# CAD primitive learning

In this notebook I will explore the second step of my proposed idea, where PN++ and Deepcad are trained together to learn extrusion primitives.

**IMPORTANT** 
For this I again turned of the random sampling in the dataset, in order to get all points.

## Dataset creation

In [29]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PCExtrusionSegmentationDataset

import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

import open3d as o3d
import torch

In [30]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)

In [31]:
data = train_dataset[5]

In [32]:
id = train_dataset.get_id(5)

In [33]:
h5_path = os.path.join("..", "data", "pc_from_vec_labels", id[:4], id + ".h5")

In [34]:
h5_path

'../data/pc_from_vec_labels/0003/00034076.h5'

In [35]:
sequences = {}
with h5py.File(h5_path) as f:
    print(f.keys())
    for k,v in f['sequences'].items():
        sequences[k] = v[:]

<KeysViewHDF5 ['labels', 'sequences']>


In [36]:
sequences

{'0': array([[  4,  -1,  -1, ...,  -1,  -1,  -1],
        [  2, 176, 128, ...,  -1,  -1,  -1],
        [  4,  -1,  -1, ...,  -1,  -1,  -1],
        ...,
        [  3,  -1,  -1, ...,  -1,  -1,  -1],
        [  3,  -1,  -1, ...,  -1,  -1,  -1],
        [  3,  -1,  -1, ...,  -1,  -1,  -1]]),
 '1': array([[  4,  -1,  -1, ...,  -1,  -1,  -1],
        [  2, 176, 128, ...,  -1,  -1,  -1],
        [  4,  -1,  -1, ...,  -1,  -1,  -1],
        ...,
        [  3,  -1,  -1, ...,  -1,  -1,  -1],
        [  3,  -1,  -1, ...,  -1,  -1,  -1],
        [  3,  -1,  -1, ...,  -1,  -1,  -1]])}

In [37]:
pc, label = data['pc'], data['label']

In [38]:
pc.shape

torch.Size([10000, 3])

In [39]:
label.shape

torch.Size([10000])

In [40]:
np.unique(label)

array([0, 1])

In [41]:
def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    colors = plt.cm.tab10(labels / labels.max())[:, :3]
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])

In [42]:
def visualize_pc(points):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    o3d.visualization.draw_geometries([pcd])

In [43]:
visualize_labeled_pc(pc, label)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [44]:
def split_pc_by_labels(pc: np.ndarray, labels: np.ndarray, num_classes: int = 10):
    class_pcs = []

    for class_id in np.unique(labels):
        class_mask = (labels == class_id)
        class_pc = pc[class_mask]
        class_pcs.append(class_pc)

    return class_pcs


In [45]:
pcs = split_pc_by_labels(pc, label)

In [46]:
for pc in pcs:
    visualize_pc(pc)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [ ]:
NEXT: CHECK IF SEQUENCES ALLIGN WITH PC